# Systematic Base Model Pipelines and Optimization Experiments

In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from src import base_model as utils

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Global Pipeline Configuration Settings

In [2]:
RANDOM_STATE = 42
ROOT_DIR = os.path.abspath("../")
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

In [3]:
# Ingest Transformed Dataset Partitions
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [4]:
# Standard Base Model Search Map Space
models_zoo = {
    "logistic_regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=2000),
    "decision_tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "xgboost": XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1)
}

In [5]:
# Define Constant Immutable Core Column Groups
dummyfy_columns = ['Card Type', 'NumOfProducts', 'Geography', 'Gender']
norm_std_columns = ['Balance', 'Point Earned', 'CreditScore', 'Age', 'Tenure', 'Satisfaction Score', 'EstimatedSalary']

## 2. Experiment 1: Primary Baseline Exploration (With Potential Feature Leakage)

In [6]:
EXP_1_NAME = "customer-churn-simple"
utils.init_mlflow_experiment(EXP_1_NAME, DB_PATH, ARTIFACTS_DIR)

2026/06/02 17:00:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/02 17:00:40 INFO mlflow.store.db.utils: Updating database tables


In [7]:
# Configure Experiment Schema Arrays
nomod_columns_exp1 = ['HasCrCard', 'IsActiveMember', 'Complain']

preprocessor_exp1 = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), dummyfy_columns),
        ('num', StandardScaler(), norm_std_columns),
        ('pass', 'passthrough', nomod_columns_exp1)
    ],
    remainder='drop'
)

In [8]:
# Run Pipeline Execution Training Loop
utils.train_and_log_models(
    models=models_zoo, preprocessor=preprocessor_exp1,
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns_exp1
)

2026/06/02 17:00:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 17:00:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/02 17:00:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 17:00:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Successfully processed and logged: logistic_regression
Successfully processed and logged: decision_tree


2026/06/02 17:00:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 17:00:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/02 17:00:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 17:00:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Successfully processed and logged: random_forest
Successfully processed and logged: xgboost


In [9]:
# Review Performance Metrics Results
utils.generate_experiment_summary(EXP_1_NAME)

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.pr_auc
3,95fba8e756974d98b8a19cc28170ed36,logistic_regression,0.9980,0.990291,1.0,0.995122,0.999786,0.999112
1,1811de1e5db945019ccd8a66e9bb16c1,random_forest,0.9980,0.990291,1.0,0.995122,0.999770,0.998888
0,0aaf390a130e49238449bad8c51a1854,xgboost,0.9980,0.990291,1.0,0.995122,0.999597,0.997947
2,613ffb6f4d174eaa99b74b1bb4ef020e,decision_tree,0.9975,0.987893,1.0,0.993910,0.998430,0.987893


> **Analysis Insight:** Perfect classification metrics suggest target leakage from the `Complain` column. Retraining the models without this feature is necessary to capture realistic real-world performance patterns.

## 3. Experiment 2: Robust Evaluation (Excluding Target Leakage Feature)

In [10]:
EXP_2_NAME = "customer-churn-simple-no-complain-feature"
utils.init_mlflow_experiment(EXP_2_NAME, DB_PATH, ARTIFACTS_DIR)

In [11]:
# Sanitize Feature Spaces
nomod_columns_exp2 = ['HasCrCard', 'IsActiveMember']

preprocessor_exp2 = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), dummyfy_columns),
        ('num', StandardScaler(), norm_std_columns),
        ('pass', 'passthrough', nomod_columns_exp2)
    ],
    remainder='drop'
)

In [12]:
# Run Sanitized Pipeline Iteration Execution
utils.train_and_log_models(
    models=models_zoo, preprocessor=preprocessor_exp2,
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns_exp2
)

2026/06/02 17:00:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 17:00:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/02 17:00:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 17:00:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Successfully processed and logged: logistic_regression
Successfully processed and logged: decision_tree


2026/06/02 17:00:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 17:00:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully processed and logged: random_forest


2026/06/02 17:00:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 17:00:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully processed and logged: xgboost


In [13]:
# Review Sanitized Performance Results
utils.generate_experiment_summary(EXP_2_NAME)

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.pr_auc
1,9cadd918282b426b93aea7862da1cfb8,random_forest,0.8755,0.795539,0.524510,0.632201,0.875317,0.733387
0,7c714a8ce827418ea683dac914de6d42,xgboost,0.8540,0.682390,0.531863,0.597796,0.849671,0.691571
3,147e81c0dcbe413592cd8a0c3a9012ec,logistic_regression,0.8530,0.743590,0.426471,0.542056,0.838354,0.649413
2,274a20c58b15433aa39d06da5aed442b,decision_tree,0.7965,0.501080,0.568627,0.532721,0.711763,0.372928
